# CNNs: BatchNorm per-channel broadcasting

**Solution notebook — Delta Drills #461**

Run the cells top-to-bottom to see the reference answer execute.


## Problem

Broadcasting aligns trailing dims — so adding a raw length-C bias to a (B, C, H, W) tensor only 'works' by accident when C equals W, and otherwise raises. Write solve(x, bias): attempt `x + bias` in a try block; in the except branch (RuntimeError), recover by reshaping the bias so it broadcasts per CHANNEL and return `('raised', x + reshaped)`. If the raw add did NOT raise, return `('no error', x + bias.reshape(1, -1, 1, 1))` — the accident case still needs the per-channel fix.


<details><summary>💡 Hint (click to reveal)</summary>

Try the raw add; on RuntimeError, `bias.reshape(1, C, 1, 1)` fixes alignment.

</details>


In [ ]:
%pip install -q numpy torch --index-url https://download.pytorch.org/whl/cpu

## Reference solution


In [ ]:
import torch

def solve(x, bias):
    C = x.shape[1]
    try:
        bad = x + bias
        return ('no error', x + bias.reshape(1, C, 1, 1))
    except RuntimeError:
        return ('raised', x + bias.reshape(1, C, 1, 1))

x = torch.ones(1, 2, 2, 3)
bias = torch.tensor([10.0, 20.0])
print(solve(x, bias))


## Why this works

The raw add only 'works' when C happens to equal W — and then it's silently WRONG (it adds per-column, not per-channel). When C≠W it raises, which is the honest failure. Either way the per-channel reshape (1, C, 1, 1) is the correct fix; the 'no error' branch is the more dangerous case.
